# Render Kade — NEONFALL (GPU gratis)

**Cara pakai (dari HP):**
1. Menu **Runtime** → **Change runtime type** → Hardware accelerator: **GPU (T4)** → Save.
2. Menu **Runtime** → **Run all**. Tunggu sampai semua cell selesai (~2–3 menit).
3. Di output cell terakhir ada **2 link catbox** (depan + three-quarter). Kirim kedua link itu ke Sitolol.

Isi: download Blender 4.0.2 + file Kade (.blend 19 MB), render 32 sample (denoise ON, 1280×720) 2 sudut, upload hasil ke catbox.


In [ ]:
!nvidia-smi --query-gpu=name --format=csv
!wget -q https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz -O /tmp/blender.tar.xz && echo BLENDER-DOWNLOADED
!tar -xf /tmp/blender.tar.xz -C /content/ && echo BLENDER-EXTRACTED && ls /content/blender-4.0.2-linux-x64/blender
!wget -q https://raw.githubusercontent.com/MrYos99/kade-render/main/kade_tahap3_gabungan_flap.blend -O /content/kade_tahap3_gabungan_flap.blend && echo BLEND-DOWNLOADED && ls -la /content/*.blend


In [ ]:
%%writefile /content/render_front32.py
# render_front32.py — NEONFALL Kade, front view, 32 samples, GPU/CUDA.
# Run: blender -b kade_tahap3_gabungan_flap.blend -P render_front32.py
import bpy
import time

prefs = bpy.context.preferences.addons['cycles'].preferences
prefs.get_devices()
print("CYCLES DEVICES:")
for d in prefs.devices:
    print("  -", d.type, d.name, "use:", d.use)
for d in prefs.devices:
    if d.type in ('CUDA', 'OPTIX', 'HIP', 'METAL', 'ONEAPI'):
        d.use = True
prefs.compute_device_type = 'CUDA'

scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.device = 'GPU'
scene.cycles.samples = 32
scene.cycles.use_denoising = True
scene.render.resolution_x = 1280
scene.render.resolution_y = 720
cam = bpy.data.objects['Cam_front']
cam.location = (0.0, -9.5, 1.0)
scene.camera = cam
scene.render.filepath = '/content/kade_gpu_front32.png'
print("ENGINE:", scene.render.engine, "| DEVICE:", scene.cycles.device)
print("RESOLUTION:", scene.render.resolution_x, "x", scene.render.resolution_y)
print("SAMPLES:", scene.cycles.samples)
print("CAMERA:", scene.camera.name, tuple(round(v, 2) for v in scene.camera.location))
t0 = time.time()
bpy.ops.render.render(animation=False, write_still=True)
print("RENDER DONE in %.1f s ->" % (time.time() - t0), scene.render.filepath)


In [ ]:
%%writefile /content/render_threequarter32.py
# render_threequarter32.py — NEONFALL Kade, three-quarter view, 32 samples, GPU/CUDA.
# Run: blender -b kade_tahap3_gabungan_flap.blend -P render_threequarter32.py
import bpy
import time

prefs = bpy.context.preferences.addons['cycles'].preferences
prefs.get_devices()
print("CYCLES DEVICES:")
for d in prefs.devices:
    print("  -", d.type, d.name, "use:", d.use)
for d in prefs.devices:
    if d.type in ('CUDA', 'OPTIX', 'HIP', 'METAL', 'ONEAPI'):
        d.use = True
prefs.compute_device_type = 'CUDA'

scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.device = 'GPU'
scene.cycles.samples = 32
scene.cycles.use_denoising = True
scene.render.resolution_x = 1280
scene.render.resolution_y = 720
cam = bpy.data.objects['Cam_threequarter']
cam.location = (5.8, -5.8, 1.2)
scene.camera = cam
scene.render.filepath = '/content/kade_gpu_threequarter32.png'
print("ENGINE:", scene.render.engine, "| DEVICE:", scene.cycles.device)
print("RESOLUTION:", scene.render.resolution_x, "x", scene.render.resolution_y)
print("SAMPLES:", scene.cycles.samples)
print("CAMERA:", scene.camera.name, tuple(round(v, 2) for v in scene.camera.location))
t0 = time.time()
bpy.ops.render.render(animation=False, write_still=True)
print("RENDER DONE in %.1f s ->" % (time.time() - t0), scene.render.filepath)


In [ ]:
!/content/blender-4.0.2-linux-x64/blender -b /content/kade_tahap3_gabungan_flap.blend -P /content/render_front32.py
!curl -s -F "reqtype=fileupload" -F "fileToUpload=@/content/kade_gpu_front32.png" https://catbox.moe/user/api.php | tee /content/front32_url.txt
!echo "FRONT-URL:" && cat /content/front32_url.txt


In [ ]:
!/content/blender-4.0.2-linux-x64/blender -b /content/kade_tahap3_gabungan_flap.blend -P /content/render_threequarter32.py
!curl -s -F "reqtype=fileupload" -F "fileToUpload=@/content/kade_gpu_threequarter32.png" https://catbox.moe/user/api.php | tee /content/threequarter32_url.txt
!echo "THREEQUARTER-URL:" && cat /content/threequarter32_url.txt
